In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ─── Data ────────────────────────────────────────────────────────
col_headers = ['Market', 'Regime', 'Models ↑/↓', 'Strategy Return', 'B&H Return', 'vs B&H', 'Max DD', 'Win Rate', 'Trades', 'Final ($)']
col_widths  = [0.07, 0.08, 0.16, 0.11, 0.10, 0.09, 0.09, 0.09, 0.07, 0.14]

rows = [
    ['US',   'HMM',           'RF / CNN',           '+7.04%',   '+31.63%',  '-24.59%', '-42.75%', '0.0%',   '4',  '10,703'],
    ['UK',   'HMM',           'CNN / XGBoost',      '+59.52%',  '+25.77%',  '+33.75%', '-14.06%', '66.7%',  '6',  '15,951'],
    ['Thai', 'ADX+ST',        'LSTM / CNN',         '+18.32%',  '+13.29%',  '+5.03%',  '-17.64%', '50.0%',  '8',  '11,831'],
    ['Gold', 'SMA200',        'CNN / Transformer',  '+273.32%', '+98.28%',  '+175.04%','-30.93%', '33.3%',  '3',  '37,331'],
    ['BTC',  'GMM',           'RF / RF',            '+166.38%', '+11.51%',  '+154.87%','-31.60%', '49.2%',  '59', '26,637'],
]

# ─── Layout ──────────────────────────────────────────────────────
ROW_H    = 0.62
HEADER_H = 0.72
FIG_W    = 18.0
N_ROWS   = len(rows)
FIG_H    = HEADER_H + N_ROWS * ROW_H + 0.20

BLACK = '#000000'
WHITE = '#ffffff'
GRAY  = '#f2f2f2'
LGRAY = '#cccccc'

market_bg = {
    'US':   GRAY,
    'UK':   WHITE,
    'Thai': GRAY,
    'Gold': WHITE,
    'BTC':  GRAY,
}

matplotlib.rcParams['font.family'] = ['DejaVu Sans']
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.set_xlim(0, 1)
ax.set_ylim(0, FIG_H)
ax.axis('off')
fig.patch.set_facecolor(WHITE)

def cx(i):
    return sum(col_widths[:i])

# ── Header ────────────────────────────────────────────────────────
hdr_top = FIG_H - 0.10
hdr_bot = hdr_top - HEADER_H

ax.add_patch(mpatches.FancyBboxPatch(
    (0, hdr_bot), 1, HEADER_H,
    boxstyle='square,pad=0', facecolor=BLACK, edgecolor=BLACK,
    linewidth=2, clip_on=False))

for ci, (label, w) in enumerate(zip(col_headers, col_widths)):
    if ci > 0:
        ax.plot([cx(ci), cx(ci)], [hdr_bot, hdr_top],
                color=WHITE, linewidth=0.8, zorder=5)
    ax.text(cx(ci) + w / 2, (hdr_top + hdr_bot) / 2, label,
            ha='center', va='center',
            fontsize=11, fontweight='bold', color=WHITE)

ax.plot([0, 1], [hdr_bot, hdr_bot], color=BLACK, linewidth=2)

# ── Rows ──────────────────────────────────────────────────────────
prev_market = None
for ri, row_data in enumerate(rows):
    ry0 = hdr_bot - ri * ROW_H
    ry1 = ry0 - ROW_H
    market = row_data[0]

    ax.add_patch(mpatches.FancyBboxPatch(
        (0, ry1), 1, ROW_H,
        boxstyle='square,pad=0', facecolor=market_bg[market],
        edgecolor='none', clip_on=False))

    is_new = (market != prev_market)
    ax.plot([0, 1], [ry0, ry0],
            color=BLACK if is_new else LGRAY,
            linewidth=1.5 if is_new else 0.5)

    for ci, (val, w) in enumerate(zip(row_data, col_widths)):
        if ci > 0:
            ax.plot([cx(ci), cx(ci)], [ry1, ry0],
                    color=LGRAY, linewidth=0.6, zorder=5)

        if ci == 0:                          # Market — bold
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 1:                        # Regime
            fw, fs, fst = 'normal', 10.5, 'normal'
        elif ci == 2:                        # Models — bold monospace
            fw, fs, fst = 'bold', 10.5, 'normal'
        elif ci == 3:                        # Strategy Return — bold
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 5:                        # vs B&H — bold, color-coded
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 9:                        # Final $ — bold
            fw, fs, fst = 'bold', 10.5, 'normal'
        else:                                # Other numeric
            fw, fs, fst = 'normal', 10.5, 'normal'

        ax.text(cx(ci) + w / 2, (ry0 + ry1) / 2, val,
                ha='center', va='center',
                fontsize=fs, fontweight=fw, fontstyle=fst, color=BLACK)

    prev_market = market

# ── Border ───────────────────────────────────────────────────────
bot = hdr_bot - N_ROWS * ROW_H
ax.plot([0, 1], [bot, bot], color=BLACK, linewidth=2)
ax.plot([0, 0], [bot, hdr_top], color=BLACK, linewidth=2)
ax.plot([1, 1], [bot, hdr_top], color=BLACK, linewidth=2)

# ── Save ─────────────────────────────────────────────────────────
out_dir  = '/Users/oattao/project/p-e/ipynb/image'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'table7_backtest_results.png')

plt.tight_layout(pad=0.1)
plt.savefig(out_path, dpi=200, bbox_inches='tight',
            facecolor=WHITE, pad_inches=0.12)
plt.show()
print(f'Saved → {out_path}')